# ACS Income



In [ ]:
import pickle
import random

import pandas as pd
from scipy.stats import pearsonr

try:
    from experiment import (
        ROOT, environment_versions,
        experiment_config, prepare_acs, run_trials, save_results, summarize_results,
    )
    from plotting import plot_comparison, plot_qq_matrix, plot_target_qq
except ImportError:
    from dataset_ranking.experiment import (
        ROOT, environment_versions,
        experiment_config, prepare_acs, run_trials, save_results, summarize_results,
    )
    from dataset_ranking.plotting import (
        plot_comparison, plot_qq_matrix, plot_target_qq,
    )


## Prepare data

In [ ]:
DATA_DIR = ROOT / "data"
RESULT_DIR = ROOT / "results"
FIGURE_DIR = RESULT_DIR / "figures"

data = prepare_acs(DATA_DIR, target_state=6)
pd.DataFrame({"label": data["labels"], "state": data["source_ids"]})


## Run experiment

In [ ]:
n_t = 30
n_k = 1000
n_test = 1000
trials = 1000
global_seed = 123
jobs = 1

data.update({
    "trials": trials,
    "seed_start": global_seed,
    "n_test": n_test,
    "n_t": n_t,
    "n_k": n_k,
})

results = run_trials(
    data,
    trials=trials,
    seed_start=global_seed,
    jobs=jobs,
    n_test=n_test,
    n_t=n_t,
    n_k=n_k,
    checkpoint_path=RESULT_DIR / "acs_income_checkpoint.pkl",
    checkpoint_every=4,
)
summary = save_results(data, results, RESULT_DIR)
summary


## QQ plots

In [ ]:
plot_target_qq(
    data["qq_data"],
    data["target_id"],
    data["qq_source_ids"],
    data["site_column"],
    data["response"],
    labels=data["qq_labels"],
    target_label="CA",
    title="Normal QQ Plots: California vs Other States",
    path=FIGURE_DIR / "qqplot_california_vs_other_states_whitened.png",
)

plot_qq_matrix(
    data["qq_data"],
    data["qq_source_ids"],
    data["site_column"],
    data["response"],
    labels=data["qq_labels"],
    title="Normal QQ Plots: ACS Income",
    path=FIGURE_DIR / "qqplot_matrix_acs_income_whitened.png",
)


## Result plot

In [ ]:
highlights = {
    "duc": {"NY", "NJ"},
    "neg_kl": {"NY"},
    "neg_score_x": {"NY", "NJ", "TX"},
}

plot_comparison(
    summary,
    "Comparison of methods for data selection: ACS Income",
    y_column="avg_rank",
    y_label="Average Rank by Test MSE (1 = Best)",
    highlight_map=highlights,
    path=FIGURE_DIR / "fig3.png",
)

trial_idx = random.Random(1).randrange(trials)
single_trial = summarize_results([results[trial_idx]], data["labels"])
no_highlights = {name: set() for name in highlights}
plot_comparison(
    single_trial,
    f"Comparison of data selection metrics: Trial {trial_idx}",
    y_column="weighted_mse",
    y_label="Test MSE",
    highlight_map=no_highlights,
    path=FIGURE_DIR / "ACS_one_trial.png",
)

summary_300 = summarize_results(results[:300], data["labels"])
plot_comparison(
    summary_300,
    "Comparison of methods for data selection: ACS Income (300 Trials)",
    y_column="avg_rank",
    y_label="Average Rank by Test MSE (1 = Best)",
    highlight_map=no_highlights,
    path=FIGURE_DIR / "acs_300.png",
)


Code for Table S1


In [ ]:
table_s1_trials = 500
table_s1_counts = [300, 400, 500]

table_s1_results = run_trials(
    data,
    trials=table_s1_trials,
    seed_start=global_seed,
    jobs=jobs,
    n_test=n_test,
    n_t=n_t,
    n_k=n_k,
    n_target_mean=50,
    checkpoint_path=RESULT_DIR / "acs_target_mean_50_main_checkpoint.pkl",
    checkpoint_every=4,
)

with open(RESULT_DIR / "acs_target_mean_50_results.pkl", "wb") as file:
    pickle.dump({
        "config": experiment_config(
            data, global_seed, table_s1_trials, n_test, n_t, n_k, 50,
        ),
        "labels": data["labels"],
        "source_ids": data["source_ids"],
        "seeds": list(range(global_seed, global_seed + table_s1_trials)),
        "results": table_s1_results,
        "target_mean_sample": 50,
        "prediction_data": "standardized",
        "baseline_data": "standardized",
        "duc_data": "whitened",
        "environment": environment_versions(),
    }, file)


In [ ]:
def table_s1_correlation(trials):
    trial_summary = summarize_results(
        table_s1_results[:trials], data["labels"]
    )
    return pearsonr(trial_summary["duc"], trial_summary["avg_rank"])[0]

table_s1 = pd.DataFrame({
    "trials": table_s1_counts,
    "correlation": [
        table_s1_correlation(trials) for trials in table_s1_counts
    ],
})
table_s1.to_csv(RESULT_DIR / "table_s1_correlations.csv", index=False)
table_s1.assign(correlation=table_s1.correlation.round(3))
